In [3]:

import pandas as pd
import numpy as np
import os

df = pd.read_csv('electrical_fault_detect_dataset.csv')
print(df.head())
print(df.describe())

   Output (S)          Ia        Ib          Ic        Va        Vb        Vc
0           0 -170.472196  9.219613  161.252583  0.054490 -0.659921  0.605431
1           0 -122.235754  6.168667  116.067087  0.102000 -0.628612  0.526202
2           0  -90.161474  3.813632   86.347841  0.141026 -0.605277  0.464251
3           0  -79.904916  2.398803   77.506112  0.156272 -0.602235  0.445963
4           0  -63.885255  0.590667   63.294587  0.180451 -0.591501  0.411050
         Output (S)            Ia            Ib            Ic            Va  \
count  12001.000000  12001.000000  12001.000000  12001.000000  12001.000000   
mean       0.457962      6.709369    -26.557793     22.353043      0.010517   
std        0.498250    377.158470    357.458613    302.052809      0.346221   
min        0.000000   -883.542316   -900.526951   -883.357762     -0.620748   
25%        0.000000    -64.348986    -51.421937    -54.562257     -0.237610   
50%        0.000000     -3.239788      4.711283     -0.399

In [ ]:
%pip install scikit-learn

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split # Standardization of the data i.e mean = 0 , std = 1 & Data scaling

electrical_cols = ['Ia' , 'Ib' , 'Ic' , 'Va' , 'Vb' , 'Vc']

X = df[electrical_cols].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

normal_mask = df['Output (S)'] == 0
X_normal = X_scaled[normal_mask]
X_train, X_val = train_test_split(X_normal, test_size=0.2, random_state=42)

In [ ]:
%pip install scipy

In [ ]:
from scipy.stats import kurtosis, skew
def extract_electrical_features(df):
    features = pd.DataFrame()

    # RMS values
    for col in ['Ia', 'Ib', 'Ic', 'Va', 'Vb', 'Vc']:
        features[f'{col}_rms'] = np.sqrt(np.mean(df[col]**2, axis=1)) \
            if df[col].ndim > 1 else np.sqrt(df[col]**2)

    # Current imbalance
    features['current_imbalance'] = (
        df[['Ia','Ib','Ic']].max(axis=1) - df[['Ia','Ib','Ic']].min(axis=1)
    ) / (df[['Ia','Ib','Ic']].mean(axis=1) + 1e-6)

    # Statistical features
    features['kurtosis_Ia'] = kurtosis(df['Ia'], axis=1) if df['Ia'].ndim > 1 else kurtosis(df['Ia'])
    features['skew_Ia'] = skew(df['Ia'], axis=1) if df['Ia'].ndim > 1 else skew(df['Ia'])

    return features.values